# 👁️ Ophthalmo-AI — Training Model Vision (Google Colab)

Notebook ini melatih model **Computer Vision** yang menjadi *Tahap 1* dari pipeline Ophthalmo-AI,
lalu menyajikannya sebagai **endpoint inference `/predict`** yang dipanggil Cloud Function `analyzeVision`.

```
Foto mata → [MODEL CV — notebook ini] → fitur visual + probabilitas kelas
          → [Claude API — ophthalmoTriage] → triase klinis final (JSON ketat)
```

**Yang Anda butuhkan sebelum menjalankan:**
1. **Runtime GPU**: menu *Runtime → Change runtime type → T4 GPU*.
2. **`kaggle.json`** (API token): kaggle.com → *Settings → API → Create New Token*.
3. **Token ngrok** (gratis, utk mengekspos endpoint): dashboard.ngrok.com → *Your Authtoken*.

**Alur:** Konfigurasi → Unduh dataset → Latih → Evaluasi → Simpan → Serving `/predict` → salin URL ke `functions/.env`.


In [ ]:
# ============================================================
# 1. KONFIGURASI — sesuaikan di sini saja
# ============================================================

# Dataset Kaggle (format: pemilik/slug). Default: klasifikasi penyakit mata
# 4 kelas (cataract / diabetic_retinopathy / glaucoma / normal).
# Ganti dgn dataset mata eksternal milik Anda bila sudah ada — struktur
# folder harus: satu subfolder per kelas berisi gambar.
KAGGLE_DATASET = "gunavenkatdoddi/eye-diseases-classification"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 8        # tahap 1: melatih kepala klasifikasi
EPOCHS_FINETUNE = 5    # tahap 2: fine-tune sebagian backbone
SEED = 42

# Pemetaan kelas dataset → fitur visual yang dipahami Ophthalmo-AI.
# Probabilitas kelas menjadi skor severity fitur terkait.
# SESUAIKAN dengan kelas dataset yang Anda pakai!
CLASS_FEATURE_MAP = {
    "cataract":              [("Kekeruhan Kornea/Lensa", 1.0)],
    "diabetic_retinopathy":  [("Indikasi Kelainan Segmen Posterior", 1.0)],
    "glaucoma":              [("Indikasi Kelainan Papil/Saraf Optik", 1.0)],
    "normal":                [],
}

# Fitur tambahan yang selalu dilaporkan 0 bila tidak ada kelas terkait
# (menjaga kontrak respons tetap konsisten dgn web app).
BASE_FEATURES = [
    "Hiperemia Konjungtiva",
    "Kekeruhan Kornea/Lensa",
    "Edema Palpebra/Periorbital",
    "Indikasi Kelainan Segmen Posterior",
    "Indikasi Kelainan Papil/Saraf Optik",
]

print("Konfigurasi siap.")


In [ ]:
# ============================================================
# 2. UNDUH DATASET VIA KAGGLE API
#    (akan meminta upload file kaggle.json)
# ============================================================
import os, json, zipfile, pathlib
from google.colab import files

os.makedirs("/root/.kaggle", exist_ok=True)
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload kaggle.json Anda:")
    up = files.upload()
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(next(iter(up.values())))
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip -q install kaggle
!kaggle datasets download -d {KAGGLE_DATASET} -p /content/data --unzip

# Cari folder yang berisi subfolder kelas
def find_class_root(base):
    base = pathlib.Path(base)
    for p in [base, *base.rglob("*")]:
        if p.is_dir():
            subs = [d for d in p.iterdir() if d.is_dir()]
            if len(subs) >= 2 and all(
                any(f.suffix.lower() in {".jpg", ".jpeg", ".png"} for f in d.iterdir() if f.is_file())
                for d in subs[:2]
            ):
                return str(p)
    raise RuntimeError("Struktur folder kelas tidak ditemukan — cek dataset.")

DATA_DIR = find_class_root("/content/data")
print("DATA_DIR:", DATA_DIR)
print("Kelas:", sorted(os.listdir(DATA_DIR)))


In [ ]:
# ============================================================
# 3. PIPELINE DATA (train/val split + augmentasi)
# ============================================================
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("Kelas terdeteksi:", CLASS_NAMES)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name="augmentasi")


In [ ]:
# ============================================================
# 4. MODEL — transfer learning EfficientNetB0
# ============================================================
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base(x, training=False)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cb = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3),
]

print("— Tahap 1: melatih kepala klasifikasi —")
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=cb)

print("— Tahap 2: fine-tune 40 lapis teratas backbone —")
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE, callbacks=cb)


In [ ]:
# ============================================================
# 5. EVALUASI — akurasi, classification report, confusion matrix
# ============================================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

y_true, y_pred = [], []
for imgs, labels in val_ds:
    p = model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(p, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("Prediksi"); plt.ylabel("Aktual"); plt.title("Confusion Matrix")
plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# 6. SIMPAN MODEL + LABEL (dan salin ke Google Drive - opsional)
# ============================================================
import json

model.save("/content/ophthalmo_vision.keras")
with open("/content/labels.json", "w") as f:
    json.dump(CLASS_NAMES, f)
print("Model tersimpan: /content/ophthalmo_vision.keras")

# Opsional — simpan permanen ke Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/ophthalmo_vision.keras /content/labels.json /content/drive/MyDrive/


## 7. Serving — Endpoint Inference `/predict`

Sel berikut menjalankan **FastAPI** di dalam Colab dan mengeksposnya lewat **ngrok**.

**Kontrak respons** (dikonsumsi Cloud Function `analyzeVision`):
```json
POST /predict  { "image": "<base64>" }
→ {
    "quality": "Baik",
    "segment": "Eksternal (kamera ponsel)",
    "detected_features": [ { "feature": "...", "severity": 0.83 }, ... ],
    "class_probabilities": { "cataract": 0.83, "normal": 0.10, ... },
    "raw_summary": "..."
  }
```

Setelah URL ngrok tampil → isi `functions/.env` di project web:
```
VISION_ENDPOINT=https://xxxx.ngrok-free.app/predict
```
lalu `firebase deploy --only functions`.

> ⚠️ URL ngrok gratis berubah setiap kali sel di-restart, dan mati saat Colab ditutup —
> cukup untuk demo/penjurian. Untuk produksi sesungguhnya, deploy model ke Cloud Run
> (lihat colab/README.md).

In [ ]:
# ============================================================
# 7. SERVING /predict + ngrok
# ============================================================
!pip -q install fastapi uvicorn pyngrok nest-asyncio pillow

import base64, io, threading, getpass
import numpy as np, tensorflow as tf
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio, uvicorn
from pyngrok import ngrok

serving_model = tf.keras.models.load_model("/content/ophthalmo_vision.keras")

app = FastAPI(title="Ophthalmo-AI Vision Endpoint")

class PredictIn(BaseModel):
    image: str  # base64 (tanpa prefix data:)

def blur_score(img_gray):
    # varian Laplacian sederhana utk deteksi blur (kualitas citra)
    lap = np.abs(np.diff(img_gray, 2, axis=0)).mean() + np.abs(np.diff(img_gray, 2, axis=1)).mean()
    return float(lap)

@app.get("/health")
def health():
    return {"ok": True, "classes": CLASS_NAMES}

@app.post("/predict")
def predict(inp: PredictIn):
    raw = base64.b64decode(inp.image)
    img = Image.open(io.BytesIO(raw)).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = np.expand_dims(np.array(img, dtype=np.float32), 0)
    probs = serving_model.predict(arr, verbose=0)[0]

    class_probs = {c: round(float(p), 4) for c, p in zip(CLASS_NAMES, probs)}

    # Peta probabilitas kelas → skor fitur visual
    feat_scores = {f: 0.0 for f in BASE_FEATURES}
    for cls, prob in class_probs.items():
        for feat, w in CLASS_FEATURE_MAP.get(cls, []):
            feat_scores[feat] = max(feat_scores.get(feat, 0.0), round(prob * w, 2))

    quality = "Baik" if blur_score(np.array(img.convert("L"), dtype=np.float32)) > 2.0 else "Kurang — citra buram"
    top = max(class_probs, key=class_probs.get)

    return {
        "quality": quality,
        "segment": "Eksternal (kamera ponsel)",
        "detected_features": [
            {"feature": f, "severity": s} for f, s in feat_scores.items()
        ],
        "class_probabilities": class_probs,
        "raw_summary": (
            f"Model CV memprediksi kelas dominan '{top}' "
            f"(prob {class_probs[top]:.2f}). Kualitas citra: {quality}."
        ),
    }

# ---- ngrok ----
token = getpass.getpass("Tempel NGROK authtoken Anda: ")
ngrok.set_auth_token(token)
public = ngrok.connect(8000)
print("\n===============================================")
print("  VISION_ENDPOINT =", public.public_url + "/predict")
print("  → salin ke functions/.env lalu deploy ulang")
print("===============================================\n")

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()
print("Server berjalan. Uji: curl " + public.public_url + "/health")


## 8. Uji Cepat Endpoint

Jalankan sel di bawah untuk mengetes `/predict` memakai satu gambar dari dataset —
responsnya harus persis mengikuti kontrak yang dibaca Cloud Function.

In [ ]:
# ============================================================
# 8. SMOKE TEST endpoint lokal
# ============================================================
import requests, base64, pathlib, json

sample = next(p for p in pathlib.Path(DATA_DIR).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
b64 = base64.b64encode(sample.read_bytes()).decode()
r = requests.post("http://127.0.0.1:8000/predict", json={"image": b64}, timeout=60)
print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))
